# Phase 6 — Sequence Dataset Construction
### Bird Migration Prediction Project | Group C | ISI Kolkata IDEAS Internship 2026

## Step 1 — Install and Import Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


## Step 2 — Load Clustered Dataset

In [2]:
df = pd.read_csv('bird_migration_clustered.csv')
df['date_time'] = pd.to_datetime(df['date_time'], utc=True)
df = df.sort_values(['bird_name', 'date_time']).reset_index(drop=True)

print(f'Shape: {df.shape}   Missing values: {df.isnull().sum().sum()}')
df['cluster_id'].value_counts(normalize=True).sort_index() * 100


Shape: (61920, 12)   Missing values: 0


cluster_id
0    30.631460
1    24.882106
2    44.486434
Name: proportion, dtype: float64

## Step 3 — Define Configuration

In [3]:
WINDOW_SIZE = 5

FEATURE_COLS = [
    'latitude',
    'longitude',
    'altitude_clipped',
    'speed_2d',
    'month',
    'hour',
    'season',
    'is_resting'
]

TARGET_COL    = 'cluster_id'
METADATA_COLS = ['date_time', 'bird_name']
BIRDS         = ['Eric', 'Nico', 'Sanne']

# Classes: 0 = North Africa Transit, 1 = Europe/Netherlands, 2 = West Africa Wintering
print(f'Window size: {WINDOW_SIZE} | Features/step: {len(FEATURE_COLS)} | Total feature cols: {WINDOW_SIZE * len(FEATURE_COLS)}')


Window size: 5 | Features/step: 8 | Total feature cols: 40


## Step 4 — Build Sliding Window Function

In [4]:
def build_windows_for_bird(bird_df, window_size, feature_cols, target_col):
    bird_df      = bird_df.reset_index(drop=True)
    feature_array = bird_df[feature_cols].values
    n            = len(bird_df)
    suffixes     = [f't-{window_size - t}' for t in range(window_size)]
    col_names    = [f'{feat}_{suf}' for suf in suffixes for feat in feature_cols]
    rows         = np.array([
        feature_array[i - window_size: i].flatten()
        for i in range(window_size, n)
    ])
    df_out = pd.DataFrame(rows, columns=col_names)
    target_rows             = bird_df.iloc[window_size:].reset_index(drop=True)
    df_out['target']        = target_rows[target_col].values
    df_out['date_time']     = target_rows['date_time'].values
    df_out['bird_name']     = target_rows['bird_name'].values
    return df_out

print('Sliding window function defined')
print(f'Each window will produce {WINDOW_SIZE * len(FEATURE_COLS)} feature columns + 1 target + 2 metadata')


Sliding window function defined
Each window will produce 40 feature columns + 1 target + 2 metadata


## Step 5 — Build Sequence Dataset Per Bird

In [5]:
all_windows = []

for bird in BIRDS:
    bird_df = df[df['bird_name'] == bird].copy()
    bird_df = bird_df.sort_values('date_time').reset_index(drop=True)
    bird_windows = build_windows_for_bird(
        bird_df,
        window_size=WINDOW_SIZE,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL
    )
    all_windows.append(bird_windows)
    print(f'{bird}: {len(bird_df):,} fixes -> {len(bird_windows):,} windows')

df_sequence = pd.concat(all_windows, ignore_index=True)
df_sequence = df_sequence.sort_values(['bird_name', 'date_time']).reset_index(drop=True)

print(f'Total: {df_sequence.shape}')


Eric: 19,795 fixes -> 19,790 windows
Nico: 21,121 fixes -> 21,116 windows
Sanne: 21,004 fixes -> 20,999 windows
Total: (61905, 43)


## Step 6 — Drop NaN Rows

In [6]:
rows_before = len(df_sequence)
df_sequence.dropna(inplace=True)
df_sequence.reset_index(drop=True, inplace=True)
rows_after = len(df_sequence)

print(f'Rows: {rows_before:,} -> {rows_after:,} ({rows_before - rows_after} dropped)')


Rows: 61,905 -> 61,905 (0 dropped)


## Step 7 — Dataset Overview

In [7]:
feature_only_cols = [c for c in df_sequence.columns if c not in ['target', 'date_time', 'bird_name']]

print(f'Rows: {len(df_sequence):,}   Columns: {len(df_sequence.columns)}   Feature cols: {len(feature_only_cols)}')
df_sequence.iloc[0][['latitude_t-5','longitude_t-5','latitude_t-1','longitude_t-1','target','bird_name','date_time']]


Rows: 61,905   Columns: 43   Feature cols: 40


latitude_t-5                49.41986
longitude_t-5               2.120733
latitude_t-1               49.420331
longitude_t-1               2.120887
target                             0
bird_name                       Eric
date_time        2013-08-15 02:47:38
Name: 0, dtype: object

## Step 8 — Target Distribution Analysis

In [8]:
phase5_dist = {0: 26.0, 1: 31.1, 2: 42.9}
phase6_dist = df_sequence['target'].value_counts(normalize=True).sort_index() * 100

cluster_labels = {
    0: 'North Africa Transit',
    1: 'Europe / Netherlands',
    2: 'West Africa Wintering'
}

for cid in [0, 1, 2]:
    count = df_sequence[df_sequence['target'] == cid].shape[0]
    print(f'{cluster_labels[cid]:<25} Phase5={phase5_dist[cid]:.1f}%  Phase6={phase6_dist[cid]:.1f}%  n={count:,}')

pd.crosstab(df_sequence['bird_name'], df_sequence['target'], normalize='index').round(3) * 100


North Africa Transit      Phase5=26.0%  Phase6=30.6%  n=18,952
Europe / Netherlands      Phase5=31.1%  Phase6=24.9%  n=15,407
West Africa Wintering     Phase5=42.9%  Phase6=44.5%  n=27,546


target,0,1,2
bird_name,,,
Eric,42.0,58.0,0.0
Nico,39.1,6.1,54.8
Sanne,11.4,12.5,76.1


## Step 9 — Visualisation: Target Distribution

In [9]:
target_counts = df_sequence['target'].value_counts().sort_index().reset_index()
target_counts.columns = ['cluster_id', 'count']
target_counts['label'] = target_counts['cluster_id'].map(cluster_labels)
target_counts['percentage'] = (target_counts['count'] / len(df_sequence) * 100).round(1)

fig = px.bar(
    target_counts,
    x='label',
    y='count',
    color='label',
    color_discrete_sequence=['#E76F51', '#065A82', '#2A9D8F'],
    text='percentage',
    title='Phase 6 — Target Variable Distribution (cluster_id)',
    labels={'label': 'Migration Zone', 'count': 'Number of Windows'}
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()
fig.write_html('phase6_target_distribution.html')

## Step 10 — Visualisation: Windows Per Bird

In [10]:
bird_counts = df_sequence['bird_name'].value_counts().reset_index()
bird_counts.columns = ['bird_name', 'count']

fig = px.bar(
    bird_counts,
    x='bird_name',
    y='count',
    color='bird_name',
    color_discrete_map={'Eric': '#065A82', 'Nico': '#02C39A', 'Sanne': '#F4A261'},
    text='count',
    title='Phase 6 — Number of Sequence Windows per Bird',
    labels={'bird_name': 'Bird', 'count': 'Number of Windows'}
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()
fig.write_html('phase6_windows_per_bird.html')

## Step 11 — Visualisation: Target Distribution Per Bird

In [11]:
bird_target = df_sequence.groupby(['bird_name', 'target']).size().reset_index(name='count')
bird_target['label'] = bird_target['target'].map(cluster_labels)

fig = px.bar(
    bird_target,
    x='bird_name',
    y='count',
    color='label',
    color_discrete_sequence=['#E76F51', '#065A82', '#2A9D8F'],
    barmode='group',
    title='Phase 6 — Target Distribution per Bird',
    labels={'bird_name': 'Bird', 'count': 'Number of Windows', 'label': 'Migration Zone'}
)
fig.update_layout(height=500)
fig.show()
fig.write_html('phase6_target_per_bird.html')

## Step 12 — Visualisation: Window Timeline

In [12]:
df_sequence['date_time_plot'] = pd.to_datetime(df_sequence['date_time'])
df_sequence['month_year'] = df_sequence['date_time_plot'].dt.to_period('M').astype(str)

monthly_target = df_sequence.groupby(['month_year', 'target']).size().reset_index(name='count')
monthly_target['label'] = monthly_target['target'].map(cluster_labels)

fig = px.bar(
    monthly_target,
    x='month_year',
    y='count',
    color='label',
    color_discrete_sequence=['#E76F51', '#065A82', '#2A9D8F'],
    title='Phase 6 — Monthly Window Count by Migration Zone',
    labels={'month_year': 'Month', 'count': 'Number of Windows', 'label': 'Migration Zone'}
)
fig.update_layout(height=500, xaxis_tickangle=45)
fig.show()
fig.write_html('phase6_timeline.html')

## Step 14 — Feature Column Summary

In [14]:
feature_only_cols = [c for c in df_sequence.columns if c not in ['target', 'date_time', 'bird_name', 'date_time_plot', 'month_year']]

for t in range(WINDOW_SIZE, 0, -1):
    step_cols = [c for c in feature_only_cols if c.endswith(f'_t-{t}')]
    print(f't-{t}: {step_cols}')

key_cols = [f'latitude_t-{WINDOW_SIZE}', f'longitude_t-{WINDOW_SIZE}',
            f'speed_2d_t-{WINDOW_SIZE}', f'is_resting_t-{WINDOW_SIZE}']
df_sequence[key_cols].describe().round(3)


t-5: ['latitude_t-5', 'longitude_t-5', 'altitude_clipped_t-5', 'speed_2d_t-5', 'month_t-5', 'hour_t-5', 'season_t-5', 'is_resting_t-5']
t-4: ['latitude_t-4', 'longitude_t-4', 'altitude_clipped_t-4', 'speed_2d_t-4', 'month_t-4', 'hour_t-4', 'season_t-4', 'is_resting_t-4']
t-3: ['latitude_t-3', 'longitude_t-3', 'altitude_clipped_t-3', 'speed_2d_t-3', 'month_t-3', 'hour_t-3', 'season_t-3', 'is_resting_t-3']
t-2: ['latitude_t-2', 'longitude_t-2', 'altitude_clipped_t-2', 'speed_2d_t-2', 'month_t-2', 'hour_t-2', 'season_t-2', 'is_resting_t-2']
t-1: ['latitude_t-1', 'longitude_t-1', 'altitude_clipped_t-1', 'speed_2d_t-1', 'month_t-1', 'hour_t-1', 'season_t-1', 'is_resting_t-1']


,latitude_t-5,longitude_t-5,speed_2d_t-5,is_resting_t-5
count,61905.000,61905.000,61905.000,61905.000
mean,30.223,-8.956,2.559,0.453
std,14.808,8.477,3.577,0.498
min,12.354,-17.626,0.000,0.000
25%,15.393,-16.761,0.410,0.000
50%,30.424,-9.663,1.209,0.000
75%,49.999,2.603,3.058,1.000
max,51.518,4.858,63.488,1.000


## Step 15 — Save Sequence Dataset

In [15]:
df_save = df_sequence.drop(columns=['date_time_plot', 'month_year'], errors='ignore')
df_save.to_csv('bird_migration_sequence.csv', index=False)

